In [1]:
!pip install opencv-python matplotlib numpy

In [ ]:
!pip install tensorflow scikit-image

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

from skimage.color import rgb2lab
from skimage.color import lab2rgb
from skimage.transform import resize

In [ ]:
MODEL_PATH = "colorize_autoencoder_first1000.keras"

model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
model.summary()

In [ ]:
def preprocess_method1(image_path):

    img = cv2.imread(image_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = resize(img, (256,256))

    gray = cv2.cvtColor(
        (img*255).astype(np.uint8),
        cv2.COLOR_RGB2GRAY
    )

    # تکرار سه کاناله
    gray_3channel = np.stack(
        (gray,)*3,
        axis=-1
    )

    # تبدیل به Lab
    lab = rgb2lab(gray_3channel)

    # استخراج L
    L = lab[:,:,0]

    # نرمال سازی
    L = L / 100.0

    # reshape
    L_input = L.reshape(
        1,256,256,1
    )

    return img, gray, L_input, L

In [ ]:
img, gray, L_input, L = preprocess_method1(
    "forest.397.jpg"
)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Original")

plt.subplot(1,3,2)
plt.imshow(gray, cmap="gray")
plt.title("Gray")

plt.subplot(1,3,3)
plt.imshow(L, cmap="gray")
plt.title("L Channel")

plt.show()

In [ ]:
output = model.predict(L_input)

In [ ]:
ab = output[0]

# بازگردانی بازه
ab = ab * 128

L_channel = L_input[0][:,:,0] * 100

Lab = np.zeros((256,256,3))

Lab[:,:,0] = L_channel
Lab[:,:,1:] = ab

colorized = lab2rgb(Lab)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(gray, cmap="gray")
plt.title("Input Gray")

plt.subplot(1,2,2)
plt.imshow(colorized)
plt.title("Colorized Output")

plt.show()

In [ ]:
def preprocess_method2(image_path):

    # خواندن مستقیم رنگی با OpenCV
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = resize(img, (256,256))

    gray = cv2.cvtColor(
        (img*255).astype(np.uint8),
        cv2.COLOR_RGB2GRAY
    )

    gray_3channel = cv2.cvtColor(
        gray,
        cv2.COLOR_GRAY2RGB
    )

    lab = rgb2lab(gray_3channel)

    L = lab[:,:,0]

    L_input = (L / 100.0).reshape(1,256,256,1)

    return img, gray, L_input, L

In [ ]:
def colorize_image(L_input):

    output = model.predict(L_input)

    ab = output[0] * 128

    L_channel = L_input[0,:,:,0] * 100

    Lab = np.zeros((256,256,3))
    Lab[:,:,0] = L_channel
    Lab[:,:,1:] = ab

    colorized = lab2rgb(Lab)

    return colorized

In [ ]:
image_path = "forest.397.jpg"

img1, gray1, L_input1, L1 = preprocess_method1(image_path)
img2, gray2, L_input2, L2 = preprocess_method2(image_path)

colorized1 = colorize_image(L_input1)
colorized2 = colorize_image(L_input2)

plt.figure(figsize=(15,5))

plt.subplot(1,4,1)
plt.imshow(img1)
plt.title("Original")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(gray1, cmap="gray")
plt.title("Gray")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(colorized1)
plt.title("Method 1")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(colorized2)
plt.title("Method 2")
plt.axis("off")

plt.show()

In [ ]:
image_paths = [
    "forest.397.jpg",
    "forest.399.jpg",
    "YIN8QU2CC97X.jpg",
    "ZYLOMHW6DWLE.jpg"
]

for image_path in image_paths:

    img1, gray1, L_input1, L1 = preprocess_method1(image_path)
    img2, gray2, L_input2, L2 = preprocess_method2(image_path)

    colorized1 = colorize_image(L_input1)
    colorized2 = colorize_image(L_input2)

    plt.figure(figsize=(15,5))

    plt.subplot(1,4,1)
    plt.imshow(img1)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(gray1, cmap="gray")
    plt.title("Gray")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(colorized1)
    plt.title("Method 1")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(colorized2)
    plt.title("Method 2")
    plt.axis("off")

    plt.suptitle(image_path)
    plt.show()

در این بخش دو روش مختلف برای آماده‌سازی تصویر خاکستری بررسی شد. در روش اول، تصویر ابتدا خاکستری شد و سپس کانال خاکستری سه بار تکرار شد تا یک تصویر سه‌کاناله ساخته شود. در روش دوم، تصویر به صورت رنگی با OpenCV خوانده شد و سپس به خاکستری تبدیل گردید. در بیشتر تصاویر خروجی دو روش بسیار نزدیک بود، زیرا در هر دو حالت در نهایت فقط کانال روشنایی L به مدل داده می‌شود. تفاوت‌های جزئی خروجی می‌تواند به نحوه تبدیل رنگ، تغییر اندازه و محاسبه کانال L مربوط باشد.